# Park fit: a batted-ball home run model

The question this project started with: **which hitters fit which parks?** A 380 ft fly ball
down the line is a home run at Yankee Stadium and an out at Oracle Park. To answer it, we need a
model that says, for one batted ball, **how likely it is to clear the wall in a given park.**
Then we can replay every ball a hitter hit into all 30 parks and add up the expected home runs.

Notebook 06 scored columns as-is (no model). **This notebook trains models.**

The plan, as a ladder from simple to complex. Every step is scored on the same held-out season:

1. Load the data
2. What's missing, and why
3. Split by season
4. How we score a model
5. Logistic regression, one feature group at a time
6. XGBoost
7. Do pitch features help?
8. The `hit_distance` trap
9. Calibration: do the probabilities mean what they say?
10. Park fit: replay every ball in every park
11. Wrap-up

Cells marked **❓** are yours to answer. Write your answer in the cell (or in your head) before
running the next code cell. Some ask you to **predict** a result before you see it.

## 1. Load the data

Read-only connection: it won't block `dbt build`, but close it (last cell) before running dbt.

One row per **regular-season fly ball or line drive**. Ground balls and popups don't leave
the park. The label is `is_over_fence_hr` (cleared the wall), same as notebook 06.

Three kinds of features, all from the marts:
- **the ball:** exit velocity, launch angle, direction (`pull_angle_deg`)
- **the park where it was hit:** wall distance and height *in the ball's direction*, elevation,
  roof, and last season's fly ball carry (`prior_extra_distance_ft`)
- **the pitch:** speed, location, spin, movement (used in section 7 only)

In [ ]:
import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

con = duckdb.connect("../data/statcast.duckdb", read_only=True)

raw = con.sql("""
    select
        f.pitch_id, f.season, f.game_date, f.batter_id, pl.full_name as batter_name,
        f.venue_id, v.venue_name, g.is_neutral_site,
        f.batter_stand, f.batted_ball_type,
        -- the ball
        f.launch_speed, f.launch_angle, f.hit_distance, f.spray_angle_deg, f.pull_angle_deg,
        -- the park, in the ball's direction
        f.wall_distance_at_spray_ft, f.wall_height_at_spray_ft,
        v.elevation_ft, v.roof_type, v.prior_extra_distance_ft,
        -- the pitch
        f.release_speed, f.plate_x, f.plate_z, s.release_spin_rate, s.pfx_x, s.pfx_z,
        -- the label
        f.is_over_fence_hr::int as hr
    from fct_batted_balls f
    join dim_games g          on g.game_pk = f.game_pk
    join dim_venue_seasons v  on v.venue_id = f.venue_id and v.season = f.season
    join stg_savant__pitches s on s.pitch_id = f.pitch_id
    join dim_players pl       on pl.player_id = f.batter_id
    where f.game_type = 'R'
      and f.batted_ball_type in ('fly_ball', 'line_drive')
""").df()

print(f"{len(raw):,} fly balls and line drives, {raw.hr.sum():,} over-the-fence home runs "
      f"({raw.hr.mean():.1%})")
raw.groupby("season").agg(balls=("hr", "size"), home_runs=("hr", "sum"))

**❓**
1. About 9% of these balls are home runs. If a model said "not a home run" every time, how
   accurate would it be? Why is that a reason not to judge these models by accuracy?
2. Every feature comes from a **mart**, never from raw files. What does the notebook get
   for free from that (think about notebook 06's reconciliation check, the contracts, and
   the leakage rule in `dim_venue_seasons`)?

## 2. What's missing, and why

Models need every feature filled in (logistic regression does; XGBoost is more forgiving,
see the stretch question). Before dropping anything, look at **what** is missing and
**why**. Missing data is rarely random.

In [ ]:
reasons = pd.DataFrame({
    "no launch data (tracking missed it)": raw.launch_speed.isna() | raw.launch_angle.isna(),
    "no direction (no hit coordinates)": raw.spray_angle_deg.isna(),
    "one-off neutral site": raw.is_neutral_site,
    "no wall (park not measured)": raw.wall_distance_at_spray_ft.isna(),
    "no prior-season carry": raw.prior_extra_distance_ft.isna(),
})
summary = pd.DataFrame({
    "balls": reasons.sum(),
    "home_runs": [raw.hr[m].sum() for m in reasons.columns.map(reasons.get)],
})
summary

In [ ]:
# Which parks have no prior-season carry, outside the neutral sites?
(raw[raw.prior_extra_distance_ft.isna() & ~raw.is_neutral_site]
    .groupby(["season", "venue_name"]).size().rename("balls").reset_index())

**❓**
1. Look at the parks with no prior-season carry that *aren't* neutral sites. Why don't they
   have a prior season? (Hint: the A's, the Rays, a hurricane.)
2. Is that missingness **random**? If you dropped those balls, which kind of park would the
   model never see in training?
3. Neutral sites (Tokyo, London, Williamsport...) host one or two games a year. Is anything
   lost by dropping them from a model that's meant to compare the 30 home parks?
4. Look at the home run column for "no direction (no hit coordinates)". What share of
   those balls are home runs, compared with about 9% overall? Hit coordinates record where
   a fielder picked the ball up. Why would a home run be the ball most likely to have
   none? What does dropping these rows do to the model's view of home runs?

**The decisions made below** (you can revisit them; that's the lesson to come on missing data):
- drop balls with no launch data, no direction, or no wall: there's nothing to model
- drop one-off neutral sites
- for the three new home parks, set `prior_extra_distance_ft` to **0** and add a flag
  `has_prior_carry = 0`

**❓** Why is 0 a sensible fill value *for this column in particular*? (Look up what
`prior_extra_distance_ft` measures in `dim_venue_seasons.yml`.) And what does the flag let
the model do that the 0 alone wouldn't?

**❓ (stretch)** XGBoost can train on rows with missing values (it learns which way to send
them at each split). Logistic regression can't. Why might you still fill in the value
yourself, even for XGBoost?

In [ ]:
df = raw[~raw.is_neutral_site].dropna(
    subset=["launch_speed", "launch_angle", "spray_angle_deg", "wall_distance_at_spray_ft"]
).copy()
df["has_prior_carry"] = df.prior_extra_distance_ft.notna().astype(int)
df["prior_extra_distance_ft"] = df.prior_extra_distance_ft.fillna(0)
df["is_dome"] = (df.roof_type == "Dome").astype(int)
df["launch_angle_sq"] = df.launch_angle ** 2

print(f"kept {len(df):,} of {len(raw):,} balls ({len(raw) - len(df):,} dropped, "
      f"{raw.hr.sum() - df.hr.sum():,} of them home runs)")

## 3. Split by season

**Train on 2024–2025. Test on 2026.** The model never sees a 2026 ball until it's scored.

**❓**
1. Why not shuffle all three seasons together and hold out a random 20%? Think about what
   two balls from the same game, the same week, or the same park-season share.
2. In real use, this model would be trained on past seasons and used on the next one. Which
   split matches that?

In [ ]:
train = df[df.season < 2026]
test = df[df.season == 2026]
print(f"train: {len(train):,} balls ({train.hr.mean():.1%} HR)   test: {len(test):,} balls ({test.hr.mean():.1%} HR)")

## 4. How we score a model

Three numbers, each answering a different question:

- **AUC:** does the model **rank** home runs above non-home runs? (Notebook 06.) Ignores
  whether the probabilities are right.
- **Log loss:** are the **probabilities** good? Punishes confident mistakes hard: saying
  1% on a ball that left the park costs a lot. Lower is better.
- **Brier score:** the mean squared error of the probabilities. Lower is better, and gentler
  on confident mistakes than log loss.

Park fit **adds up** probabilities (10 balls at 30% = 3 expected home runs), so log loss and
Brier matter as much as AUC here.

**❓** A model that always predicts the training HR rate (about 9%) has an AUC of exactly
0.5. Why? What would its log loss and Brier be good for, then?

In [ ]:
results = []

def score(name, p, data=test):
    results.append({
        "model": name,
        "AUC": roc_auc_score(data.hr, p),
        "log loss": log_loss(data.hr, p),
        "Brier": brier_score_loss(data.hr, p),
        "expected HR": p.sum(),
        "actual HR": data.hr.sum(),
    })
    return pd.DataFrame(results).set_index("model").round(4)

score("base rate (always 9%)", np.full(len(test), train.hr.mean()))

## 5. Logistic regression, one feature group at a time

Logistic regression fits one weight per feature and squeezes the weighted sum through an
S-curve into a probability. Features are standardized first so the weights are comparable.

**Step 1: exit velocity and launch angle only.**

**❓ Predict first:** these two numbers alone describe how hard and how high the ball was hit.
What AUC do you expect: 0.6, 0.75, 0.9, or 0.98?

In [ ]:
def fit_logistic(features):
    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
    model.fit(train[features], train.hr)
    return model

ball = ["launch_speed", "launch_angle"]
m1 = fit_logistic(ball)
score("LR: exit velo + launch angle", m1.predict_proba(test[ball])[:, 1])

Now look at how the home run rate actually depends on launch angle:

In [ ]:
by_angle = (train.assign(angle_bin=(train.launch_angle // 5) * 5)
            .groupby("angle_bin").hr.mean())
ax = by_angle.plot(marker="o", figsize=(7, 3.5))
ax.set(xlabel="launch angle (5° bins)", ylabel="home run rate", title="Home run rate by launch angle (2024-25)")
plt.show()

**❓**
1. Describe the shape. Where's the peak? What happens to a ball hit at 50°?
2. Logistic regression gives launch angle **one** weight: higher angle always pushes the
   probability the same way. How badly does that fit the shape above?
3. Adding `launch_angle_sq` (angle squared) lets the model bend that line into a hill.
   **Predict** how much AUC improves before running the next cell.

In [ ]:
ball2 = ["launch_speed", "launch_angle", "launch_angle_sq"]
m2 = fit_logistic(ball2)
score("LR: + launch angle²", m2.predict_proba(test[ball2])[:, 1])

**Step 3: add direction and the wall in that direction.** This is where the park enters.

In [ ]:
direction_wall = ball2 + ["pull_angle_deg", "wall_distance_at_spray_ft", "wall_height_at_spray_ft"]
m3 = fit_logistic(direction_wall)
score("LR: + direction and wall", m3.predict_proba(test[direction_wall])[:, 1])

**Step 4: add how far balls carry in the park** (elevation, dome, last season's carry).

In [ ]:
park = direction_wall + ["elevation_ft", "is_dome", "prior_extra_distance_ft", "has_prior_carry"]
m4 = fit_logistic(park)
score("LR: + park carry", m4.predict_proba(test[park])[:, 1])

In [ ]:
# The fitted weights (on standardized features: comparable sizes)
pd.Series(m4[-1].coef_[0], index=park).sort_values().round(2)

**❓**
1. Which step helped most? Which barely moved AUC? Did log loss tell the same story?
2. Coors Field is a mile high, and balls carry there. So why might the carry features add so
   little **on top of** exit velocity, angle and the wall? (Hint: is a park's carry the
   same for every ball hit there, or does the ball already "know" something?)
3. Read the weights. Is `wall_distance_at_spray_ft` negative? Should it be? What about
   `launch_angle_sq`?

## 6. XGBoost

XGBoost builds hundreds of small decision trees, each one fixing the mistakes of the ones
before it (gradient boosting). Trees split on thresholds ("launch angle between 22 and 34,
exit velo over 101"), so they find bends and interactions **without** being told about
`launch_angle_sq`.

Same features as the last logistic model, minus the squared term.

In [ ]:
park_xgb = ["launch_speed", "launch_angle", "pull_angle_deg",
            "wall_distance_at_spray_ft", "wall_height_at_spray_ft",
            "elevation_ft", "is_dome", "prior_extra_distance_ft", "has_prior_carry"]

def fit_xgb(features):
    model = xgb.XGBClassifier(
        n_estimators=400, max_depth=5, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, random_state=0,
    )
    model.fit(train[features], train.hr)
    return model

g1 = fit_xgb(park_xgb)
score("XGBoost: park", g1.predict_proba(test[park_xgb])[:, 1])

In [ ]:
# Which features did the trees lean on? (gain: how much each one improved the splits it was used in)
pd.Series(g1.get_booster().get_score(importance_type="gain")).sort_values(ascending=False).round(1)

**❓**
1. How much better is XGBoost than the best logistic regression, in AUC and in log loss?
   Is that gap big or small, given where each started?
2. Logistic regression needed `launch_angle_sq` written by hand. What's an
   **interaction** the trees can find that the logistic model can't? (Think: does an extra
   5 mph of exit velocity matter as much at 10° as at 28°?)
3. If XGBoost is only a little better, when would you still choose logistic regression?
   (Think about explaining a weight to someone, and about notebook 08.)

## 7. Do pitch features help?

You asked whether pitch type, location, spin and movement should be features. Here's the test.

**❓ Predict first:** adding pitch speed, location, spin and movement will raise AUC by
(a) a lot, (b) a little, (c) nothing. Write down your guess **and your reason** before running.

In [ ]:
park_pitch = park_xgb + ["release_speed", "plate_x", "plate_z", "release_spin_rate", "pfx_x", "pfx_z"]
g2 = fit_xgb(park_pitch)
score("XGBoost: park + pitch", g2.predict_proba(test[park_pitch])[:, 1])

**❓**
1. Were you right? Explain the result in one sentence, using the idea that exit velocity
   and launch angle are measured **after** the bat hits the ball.
2. Where **would** pitch features matter a lot? (Hint: the per-PA model, Level 2, predicts
   before the pitch is thrown, or before contact.)

## 8. The `hit_distance` trap

`hit_distance` is the best single column in notebook 06 (AUC 0.973 on its own). Add it:

In [ ]:
leaky = park_xgb + ["hit_distance"]
g3 = fit_xgb(leaky)
score("XGBoost: park + hit_distance (don't use)", g3.predict_proba(test[leaky])[:, 1])

**❓** The numbers look amazing. Here's why this model is useless for park fit:

1. `hit_distance` is where the ball **landed** (or would have). Was it measured before or
   after the ball flew through *this* park's air?
2. In section 10 we'll replay a ball hit at sea level into Coors Field. With
   `hit_distance` as a feature, would the replayed ball fly any farther in Coors? What
   would the model then say about Coors vs every other park?
3. Notebook 06 used `distance_past_wall_ft` happily. Why was it fine **there** and not
   **here**? (What question was notebook 06 asking?)

## 9. Calibration: do the probabilities mean what they say?

If the model says 30% for 1,000 balls, about 300 should be home runs. Bin the test balls by
predicted probability and compare with what happened:

In [ ]:
def calibration(p, y, bins=10):
    cuts = pd.qcut(p, q=bins, duplicates="drop")
    return pd.DataFrame({"predicted": p, "actual": y}).groupby(cuts, observed=True).mean()

fig, ax = plt.subplots(figsize=(5, 5))
for name, model, feats in [("logistic (park)", m4, park), ("XGBoost (park)", g1, park_xgb)]:
    p = model.predict_proba(test[feats])[:, 1]
    c = calibration(p, test.hr.to_numpy(), bins=20)
    ax.plot(c.predicted, c.actual, marker="o", label=name)
ax.plot([0, 1], [0, 1], color="grey", linestyle="--", label="perfect")
ax.set(xlabel="predicted HR probability", ylabel="actual HR rate", title="Calibration, 2026")
ax.legend()
plt.show()

pd.DataFrame(results).set_index("model")[["expected HR", "actual HR"]].round(0)

**❓**
1. Where does each model sit above or below the line? Which one would you trust to **add
   up** expected home runs?
2. Both models expect a few percent **fewer** 2026 home runs than happened. They were
   trained on 2024–25. Name two things that could make 2026 balls leave the park more often
   than the same exit velocity and angle did before. Is that a modeling bug, or a fact about
   the world the model can't see?

## 10. Park fit: replay every ball in every park

Now the payoff. Take a hitter's 2026 balls. Keep everything about **the ball** (exit
velocity, launch angle, direction). Swap in **another park**: its wall distance and height
in that direction, its elevation, roof and carry. Ask the model for each ball's probability
and add them up: **expected home runs if he'd hit those balls in that park.**

The wall outlines come from `app_park_walls` (built by the same `wall_at_angle` macro as
`fct_batted_balls`), one point per degree; `np.interp` fills in between.

In [ ]:
walls = con.sql("""
    select venue_id, venue_name, angle_deg, wall_distance_ft, wall_height_ft
    from app_park_walls where season = 2026 order by venue_id, angle_deg
""").df()
parks = con.sql("""
    select venue_id, elevation_ft, roof_type, prior_extra_distance_ft
    from dim_venue_seasons
    where season = 2026 and venue_id in (select venue_id from app_park_walls where season = 2026)
""").df()
parks["has_prior_carry"] = parks.prior_extra_distance_ft.notna().astype(int)
parks["prior_extra_distance_ft"] = parks.prior_extra_distance_ft.fillna(0)
parks["is_dome"] = (parks.roof_type == "Dome").astype(int)
parks = parks.set_index("venue_id")

def replay(balls, model=g1, features=park_xgb):
    """Expected home runs for these balls in each of the 30 parks."""
    angle = balls.spray_angle_deg.clip(-45, 45)
    out = []
    for venue_id, w in walls.groupby("venue_id"):
        moved = balls.copy()
        moved["wall_distance_at_spray_ft"] = np.interp(angle, w.angle_deg, w.wall_distance_ft)
        moved["wall_height_at_spray_ft"] = np.interp(angle, w.angle_deg, w.wall_height_ft)
        for col in ["elevation_ft", "is_dome", "prior_extra_distance_ft", "has_prior_carry"]:
            moved[col] = parks.at[venue_id, col]
        out.append({"park": w.venue_name.iloc[0],
                    "expected_hr": model.predict_proba(moved[features])[:, 1].sum()})
    return pd.DataFrame(out).sort_values("expected_hr", ascending=False).reset_index(drop=True)

jazz = test[test.batter_name == "Jazz Chisholm Jr."]
print(f"Jazz Chisholm Jr., 2026: {len(jazz)} fly balls and line drives, {jazz.hr.sum()} over-the-fence HR, "
      f"{g1.predict_proba(jazz[park_xgb])[:, 1].sum():.1f} expected in the parks he actually played in")
jazz_fit = replay(jazz)
jazz_fit.round(1)

In [ ]:
ax = jazz_fit.sort_values("expected_hr").plot.barh(x="park", y="expected_hr", figsize=(7, 8), legend=False)
ax.set(xlabel="expected home runs on his 2026 fly balls and line drives", ylabel="",
       title="Jazz Chisholm Jr., 2026: the same balls in every park")
plt.show()

**❓**
1. Jazz bats left-handed and pulls the ball. Do the top and bottom parks make sense for a
   lefty pull hitter? Check the right field walls in the app's spray page.
2. Compare "expected in the parks he actually played in" with his actual total. Is the
   model's own-park number close? Why is that a good sanity check **before** trusting the
   replay?
3. Coors Field has the thinnest air in baseball, yet it lands in the middle for Jazz.
   Look up Coors' wall distances in `dim_venue_seasons`. What cancels out the carry?
4. Compare this ranking with the app's rough "would clear" chart for Jazz. Where do they
   disagree, and which of the app's simplifications (no wall height, no carry) explains it?

Now every regular with enough balls: which hitters gain or lose the most depending on the
park? (This takes a few seconds: about 300 hitters × 30 parks.)

In [ ]:
regulars = test.groupby("batter_id").filter(lambda b: len(b) >= 150)
rows = []
for (batter_id, name), balls in regulars.groupby(["batter_id", "batter_name"]):
    fit = replay(balls)
    rows.append({
        "batter": name,
        "balls": len(balls),
        "actual_hr": balls.hr.sum(),
        "best_park": fit.park.iloc[0], "best_xhr": fit.expected_hr.iloc[0],
        "worst_park": fit.park.iloc[-1], "worst_xhr": fit.expected_hr.iloc[-1],
    })
fits = pd.DataFrame(rows)
fits["swing"] = fits.best_xhr - fits.worst_xhr
fits.sort_values("swing", ascending=False).head(15).round(1)

In [ ]:
# How often is each park someone's best fit, or worst?
pd.DataFrame({
    "best for": fits.best_park.value_counts(),
    "worst for": fits.worst_park.value_counts(),
}).fillna(0).astype(int).sort_values("best for", ascending=False)

**❓**
1. Who has the biggest swing between best and worst park? What kind of hitter is it:
   handedness, power, pull tendency?
2. A few parks are almost everyone's best or worst fit. Is "park fit" then mostly about the
   **park**, or the **hitter**? How would you measure fit so a hitter-friendly park
   doesn't win for everyone? (Hint: compare each hitter's best park with the average
   hitter's.)
3. The replay keeps each ball's direction and launch exactly as it was. A hitter traded to
   Yankee Stadium might change his swing to aim at the short porch. Is that a flaw in the
   model, or a limit on what question it can answer?

## 11. Wrap-up

**❓** In your own words, three or four sentences:
1. What does this model predict, and what is it **not** allowed to see?
2. Which single change helped most, and why?
3. What's the one assumption in the park-fit replay you trust least?

**Next:**
- **Notebook 08:** season home run totals and Jazz's road to 100. That's a regression
  problem where logistic and linear models are hard to beat, plus a backtest of milestone
  predictions.
- **Later:** the missing-data lesson (drop vs fill vs flag, using the new parks and the
  neutral sites from section 2).

In [ ]:
con.close()